In [8]:
from langchain_community.document_loaders import WikipediaLoader, Docx2txtLoader, PyPDFLoader, TextLoader, DirectoryLoader, WebBaseLoader
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings


import getpass


OPENAI_API_KEY = getpass.getpass('Enter your OPENAI_API_KEY')

Enter your OPENAI_API_KEY ········


In [9]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
embeddings_model = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)
vector_db = Chroma("tourist_info", embeddings_model)

In [10]:
def split_and_import(loader):
    chunks = text_splitter.split_documents(loader.load())
    vector_db.add_documents(chunks)
    print(f"Ingested chunks created by {loader}")

In [11]:
webbase_loader = WebBaseLoader(web_paths=("https://en.wikipedia.org/wiki/Paestum",))
split_and_import(webbase_loader)
word_loader = Docx2txtLoader("Paestum/Paestum-Britannica.docx")
split_and_import(word_loader)
pdf_loader = PyPDFLoader("Paestum/PaestumRevisited.pdf")
split_and_import(pdf_loader)
txt_loader = TextLoader("Paestum/Paestum-Encyclopedia.txt")
split_and_import(txt_loader)

Ingested chunks created by <langchain_community.document_loaders.web_base.WebBaseLoader object at 0x000002BB9EAA9F90>
Ingested chunks created by <langchain_community.document_loaders.word_document.Docx2txtLoader object at 0x000002BB9CE33230>
Ingested chunks created by <langchain_community.document_loaders.pdf.PyPDFLoader object at 0x000002BB9CE32F90>
Ingested chunks created by <langchain_community.document_loaders.text.TextLoader object at 0x000002BBA0787CB0>


In [12]:
loader_classes = {
    'docx': Docx2txtLoader,
    'pdf': PyPDFLoader,
    'txt': TextLoader
}

In [13]:
import os
def get_loader(filename):
    _, file_extension = os.path.splitext(filename)
    file_extension = file_extension.lstrip('.')
    loader_class = loader_classes.get(file_extension)

    if loader_class:
        return loader_class(filename)
    else:
        raise ValueError(f"No loader available for file extension '{file_extension}'")

In [16]:
folder_path = "CilentoTouristInfo"
for filename in os.listdir(folder_path): #B iterate over the files in the path
    file_path = os.path.join(folder_path, filename) #C Construct the full path to the file
   
    if os.path.isfile(file_path): #D Check if it is a file (not a directory)
        try:
            loader = get_loader(file_path) #E Instantiate the correct loader for the file
            print(f"Loader for {filename}: {loader}")
            split_and_import(loader) #F Split and ingest
        except ValueError as e:
            print(e)

Loader for Acciaroli.pdf: <langchain_community.document_loaders.pdf.PyPDFLoader object at 0x000002BBA03FCF50>
Ingested chunks created by <langchain_community.document_loaders.pdf.PyPDFLoader object at 0x000002BBA03FCF50>
Loader for Cape Palinuro.txt: <langchain_community.document_loaders.text.TextLoader object at 0x000002BB9EBB5BD0>
Ingested chunks created by <langchain_community.document_loaders.text.TextLoader object at 0x000002BB9EBB5BD0>
Loader for Casalvelino.txt: <langchain_community.document_loaders.text.TextLoader object at 0x000002BB9EBB5A90>
Ingested chunks created by <langchain_community.document_loaders.text.TextLoader object at 0x000002BB9EBB5A90>
Loader for Cilentan coast.docx: <langchain_community.document_loaders.word_document.Docx2txtLoader object at 0x000002BB9EBB5BD0>
Ingested chunks created by <langchain_community.document_loaders.word_document.Docx2txtLoader object at 0x000002BB9EBB5BD0>
Loader for Cilento Coast Map and Travel Guide.docx: <langchain_community.docum

In [12]:
folder_path = "CilentoTouristInfo"
pattern = "**/*.{docx,pdf,txt}" #A Pattern to match .docx, .pdf, and .txt files

directory_loader = DirectoryLoader(folder_path, pattern) #B Initialize the DirectoryLoader with the folder path and pattern
split_and_import(directory_loader)

ValueError: Expected Embeddings to be non-empty list or numpy array, got [] in upsert.

In [17]:
query = "Where was Poseidonia and who renamed it to Paestum?" 
results = vector_db.similarity_search(query, 4) # four clostest results
print(results)

[Document(id='86736bd3-bf91-49e9-b326-429cbc7a1a36', metadata={'source': 'https://en.wikipedia.org/wiki/Paestum', 'language': 'en', 'title': 'Paestum - Wikipedia'}, page_content='The Greek settlers who founded the city originally named it Poseidonia (Ancient Greek: Ποσειδωνία). It was eventually conquered by the local Lucanians and later the Romans. The Lucanians renamed it to Paistos and the Romans gave the city its current name.[5]\nAncient ruins and features[edit]\nAerial view of Paestum, looking north; two Hera Temples in foreground, Athena Temple in background.'), Document(id='de7c4203-2a9f-46ea-823a-7ce07ab54505', metadata={'source': 'Paestum/Paestum-Britannica.docx'}, page_content='Poseidonia was probably founded about 600\xa0BC\xa0by Greek colonists from\xa0Sybaris, along the\xa0Gulf of Taranto, and it had become a flourishing town by 540, judging from its temples. After many years’ resistance the city came under the domination of the\xa0Lucanians\xa0(an\xa0indigenous\xa0Italic

In [18]:
len(results)

4

In [19]:
from langchain_core.prompts import PromptTemplate
rag_prompt_template = """Use the following pieces of context
to answer the question at the end.
If you don't know the answer, just say that you don't know,
don't try to make up an answer.
Use three sentences maximum and keep the
answer as concise as possible.
{context}
Question: {question}
Helpful Answer:"""
rag_prompt = PromptTemplate.from_template(rag_prompt_template)


In [20]:
retriever = vector_db.as_retriever()
from langchain_core.runnables import RunnablePassthrough
question_feeder = RunnablePassthrough()
from langchain_openai import ChatOpenAI
chatbot = ChatOpenAI(openai_api_key=OPENAI_API_KEY,model_name="gpt-5-nano")

rag_chain = {"context": retriever,"question": question_feeder}|rag_prompt|chatbot

def execute_chain(chain, question):
    answer = chain.invoke(question)
    return answer

question = """Where was Poseidonia and who renamed it to Paestum. Also tell me the source."""
answer = execute_chain(rag_chain, question)
print(answer.content)

Poseidonia was founded by Greek colonists from Sybaris along the Gulf of Taranto in southern Italy. It was renamed to Paestum by the Romans (the Lucanians had earlier renamed it to Paistos). Source: Paestum - Wikipedia.


In [22]:
question = """And then, what they do?
Tell me only if you know.
Also tell me the source"""
answer = execute_chain(rag_chain, question)
print(answer.content)

They present two paths: The Way of Truth explains what is; The Way of Opinion covers appearances and mortal beliefs. The general plan is known from Sextus Empiricus and Simplicius, while The Way of Opinion survives only in fragments. Source: CilentoTouristInfo\Parmenides.docx.


In [24]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
rag_prompt_template = """Use the following pieces of context
to answer the question at the end.
If you don't know the answer, just say that you don't know,
don't try to make up an answer.
Use three sentences maximum and keep the
answer as concise as possible.
{context}
Question: {question}
Helpful Answer:"""
rag_prompt = PromptTemplate.from_template(rag_prompt_template)
retriever = vector_db.as_retriever()
question_feeder = RunnablePassthrough()
chatbot = ChatOpenAI(openai_api_key=OPENAI_API_KEY,model_name="gpt-5-nano")
rag_chain = {"context": retriever,"question": question_feeder}|rag_prompt|chatbot
def execute_chain(chain, question):
    answer = rag_chain.invoke(question)
    return answer

In [27]:
from langchain_core.prompts import ChatPromptTemplate
rag_prompt = ChatPromptTemplate.from_messages(
[
("system", """You are a helpful assistant, world-class
expert in Roman and Greek history, especially in towns
located in southern Italy. Provide interesting insights
on local history and recommend places to visit with
knowledgeable and engaging answers. Answer all questions
to the best of your ability, but only use what has been
provided in the context. If you don't know, just say you
don't know. Use three sentences maximum and keep
the answer as concise as possible."""),
("placeholder", "{chat_history_messages}"),
("assistant", "{retrieved_context}"),
("human", "{question}"),
]
)


In [28]:
from langchain_community.chat_message_histories import ChatMessageHistory
chat_history_memory = ChatMessageHistory()

def execute_chain_with_memory(chain, question):
    chat_history_memory.add_user_message(question)
    answer = chain.invoke(question)
    chat_history_memory.add_ai_message(answer)
    print(f'Full chat message history: {chat_history_memory.messages}\n\n')
    return answer

In [23]:
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables import RunnableLambda
rag_prompt = ChatPromptTemplate.from_messages(
[
("system", """You are a helpful assistant, world-class
expert in Roman and Greek history, especially in towns
located in southern Italy. Provide interesting insights
on local history and recommend places to visit with
knowledgeable and engaging answers. Answer all questions
to the best of your ability, but only use what has been
provided in the context. If you don't know, just say
you don't know. Use three sentences maximum and keep
the answer as concise as possible."""),
("placeholder", "{chat_history_messages}"),
("assistant", "{retrieved_context}"),
("human", "{question}"),
])
retriever = vector_db.as_retriever()
question_feeder = RunnablePassthrough()
chatbot = ChatOpenAI(openai_api_key=OPENAI_API_KEY, model_name="gpt-5-nano")
chat_history_memory = ChatMessageHistory()
def get_messages(x):
    return chat_history_memory.messages
rag_chain = {
    "retrieved_context": retriever,
    "question": question_feeder,
    "chat_history_messages": RunnableLambda(get_messages)
    } | rag_prompt | chatbot
def execute_chain_with_memory(chain, question):
    chat_history_memory.add_user_message(question)
    answer = chain.invoke(question)
    chat_history_memory.add_ai_message(answer)
    print(f'Full chat message history: {chat_history_memory.messages}\n\n')
    return answer

question = """Where was Poseidonia and who renamed it to Paestum? Also tell me the source."""
answer = execute_chain_with_memory(rag_chain, question)
print(answer.content)

Full chat message history: [HumanMessage(content='Where was Poseidonia and who renamed it to Paestum? Also tell me the source.', additional_kwargs={}, response_metadata={}), AIMessage(content='Poseidonia was in southern Italy, on the Gulf of Taranto. It was renamed Paistos by the Lucanians, and the Romans later gave the city its current name Paestum. Source: Paestum - Wikipedia (https://en.wikipedia.org/wiki/Paestum).', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 2756, 'prompt_tokens': 873, 'total_tokens': 3629, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 2688, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E7f7RJ65VAhWjWTCKGZBHHMVkRVZJ', 'service_tier': 'default', 'finish_reason': 'stop', '

In [24]:
question = """And then what did they do?
Also tell me the source"""
answer = execute_chain_with_memory(rag_chain, question)
print(answer.content)

Full chat message history: [HumanMessage(content='Where was Poseidonia and who renamed it to Paestum? Also tell me the source.', additional_kwargs={}, response_metadata={}), AIMessage(content='Poseidonia was in southern Italy, on the Gulf of Taranto. It was renamed Paistos by the Lucanians, and the Romans later gave the city its current name Paestum. Source: Paestum - Wikipedia (https://en.wikipedia.org/wiki/Paestum).', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 2756, 'prompt_tokens': 873, 'total_tokens': 3629, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 2688, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E7f7RJ65VAhWjWTCKGZBHHMVkRVZJ', 'service_tier': 'default', 'finish_reason': 'stop', '